In [10]:
from pyspark.sql import SparkSession

jar_paths = [
    "/home/jovyan/work/jars/delta-spark_2.12-3.1.0.jar",
    "/home/jovyan/work/jars/delta-storage-3.1.0.jar",
    "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
]
jars_string = ",".join(jar_paths)

spark = SparkSession.builder \
    .appName("LakehouseSetup_Offline") \
    .config("spark.jars", jars_string) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("Connected successfully in Offline Mode! Ready to build the Lakehouse.")

Spark Version: 3.5.0
Connected successfully in Offline Mode! Ready to build the Lakehouse.


In [2]:
# from pyspark.sql.functions import col, concat, lit, substring, rand, round, expr, monotonically_increasing_id

# print("--- Generating Synthetic Data into Landing Zone ---")

# # 1. Read customers directly from the landing zone CSV
# df_customers = spark.read.csv("s3a://olist-data/landing/olist_customers_dataset.csv", header=True, inferSchema=True)
# df_unique_customers = df_customers.select("customer_unique_id").dropDuplicates()

# # 2. Generate CRM Data
# df_crm_raw = df_unique_customers \
#     .withColumn("email", concat(lit("user_"), substring(col("customer_unique_id"), 1, 8), lit("@gmail.com"))) \
#     .withColumn("phone", concat(lit("+55-"), round(rand() * 90000 + 10000, 0).cast("int"), lit("-"), round(rand() * 9000 + 1000, 0).cast("int")))

# # FIXED: Added .coalesce(1) to force a single CSV file output
# df_crm_raw.coalesce(1).write.csv("s3a://olist-data/landing/crm", mode="overwrite", header=True)
# print("CRM Raw CSV saved to Landing Zone.")

# # 3. Generate Zendesk Data (Random 20% of customers)
# df_zendesk_raw = df_crm_raw.sample(withReplacement=False, fraction=0.20, seed=42).select("email") \
#     .withColumn("ticket_id", concat(lit("TKT-"), monotonically_increasing_id().cast("string"))) \
#     .withColumn("issue_type", expr("CASE WHEN rand() < 0.5 THEN 'Late Delivery' ELSE 'Damaged Item' END")) \
#     .withColumn("satisfaction_rating", expr("CAST(round(rand() * 4 + 1, 0) AS INT)"))

# # FIXED: Added .coalesce(1) to force a single CSV file output
# df_zendesk_raw.coalesce(1).write.csv("s3a://olist-data/landing/helpdesk", mode="overwrite", header=True)
# print("Zendesk Raw CSV saved to Landing Zone.")

--- Generating Synthetic Data into Landing Zone ---
CRM Raw CSV saved to Landing Zone.
Zendesk Raw CSV saved to Landing Zone.


In [5]:
# Run this in the first cell of your notebook
!pip install s3fs boto3

INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 2.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 11.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import boto3

# 1. CONNECTION - Use 'minio' if running inside Docker, or 'localhost' if outside

# 1. UPDATED CONNECTION - Check these three lines carefully!
ACCESS_KEY = 'admin' # Default is 'minioadmin'
SECRET_KEY = 'password' # Default is 'minioadmin'
ENDPOINT = 'http://minio:9000' # Inside Docker, use the service name 'minio'

s3 = boto3.client('s3',
    endpoint_url=ENDPOINT, 
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name='us-east-1'
)

BUCKET = "olist-data"
PREFIX = "landing/"

# 2. MAPPING - Based on your previous pipeline_tables
# Filename keyword -> Target Folder
move_map = {
    "orders": "orders/",
    "customers": "customers/",
    "items": "items/",
    "products": "products/",
    "payments": "payments/",
    "reviews": "reviews/",
    "sellers": "sellers/",
    "geolocation": "geolocation/",
    "translation": "translation/",
    "crm_identities": "crm/",
    "helpdesk_tickets": "helpdesk/"
}

print("--- Organizing MinIO Landing Zone ---")

# 3. LIST FILES IN THE ROOT OF LANDING/
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter='/')

if 'Contents' not in response:
    print("No files found in the root of the landing bucket.")
else:
    for obj in response['Contents']:
        old_key = obj['Key']
        filename = old_key.split('/')[-1]
        
        # Skip if it's just the folder itself or empty
        if not filename:
            continue
            
        # 4. MATCH AND MOVE
        moved = False
        for keyword, folder in move_map.items():
            if keyword in filename:
                new_key = f"{PREFIX}{folder}{filename}"
                
                # Copy then Delete (Standard S3 "Move" operation)
                s3.copy_object(Bucket=BUCKET, CopySource={'Bucket': BUCKET, 'Key': old_key}, Key=new_key)
                s3.delete_object(Bucket=BUCKET, Key=old_key)
                
                print(f"Moved: {filename} ➔ {folder}")
                moved = True
                break
        
        if not moved:
            print(f"No folder found for: {filename}")

print("--- Landing Zone Ready for Bronze Ingestion! ---")

--- Organizing MinIO Landing Zone ---
✅ Moved: crm_identities.csv ➔ crm/
✅ Moved: helpdesk_tickets.csv ➔ helpdesk/
✅ Moved: olist_customers_dataset.csv ➔ customers/
✅ Moved: olist_geolocation_dataset.csv ➔ geolocation/
✅ Moved: olist_order_items_dataset.csv ➔ items/
✅ Moved: olist_order_payments_dataset.csv ➔ payments/
✅ Moved: olist_order_reviews_dataset.csv ➔ reviews/
✅ Moved: olist_orders_dataset.csv ➔ orders/
✅ Moved: olist_products_dataset.csv ➔ products/
✅ Moved: olist_sellers_dataset.csv ➔ sellers/
✅ Moved: product_category_name_translation.csv ➔ translation/
--- Landing Zone Ready for Bronze Ingestion! ---


In [11]:
import boto3
from pyspark.sql.functions import current_timestamp, input_file_name
# Assuming get_spark_session is defined in your notebook or imported
# from spark_utils import get_spark_session
# spark = get_spark_session("Cloud_Native_Bronze_Ingest")

# 1. CLOUD CONFIGURATION
BUCKET = "olist-data"
LANDING_PREFIX = "landing"
ARCHIVE_PREFIX = "archive"
BRONZE_BASE_PATH = f"s3a://{BUCKET}/bronze"

# Connect to MinIO via Boto3
s3 = boto3.client('s3',
    endpoint_url='http://minio:9000', # Use 'localhost:9000' if running outside Docker
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    region_name='us-east-1'
)

# Mapping: Table Name -> Subfolder Name
pipeline_tables = {
    "olist_orders": "orders",
    "olist_customers": "customers",
    "olist_items": "items",
    "olist_products": "products",
    "olist_payments": "payments",
    "olist_reviews": "reviews",
    "olist_sellers": "sellers",
    "olist_geolocation": "geolocation",
    "translation": "translation",
    "crm_identities": "crm",
    "helpdesk_tickets": "helpdesk"
}

print("--- Starting Cloud-Native Bronze Ingestion ---")

for table_name, folder_name in pipeline_tables.items():
    landing_folder_key = f"{LANDING_PREFIX}/{folder_name}/"
    bronze_dest = f"{BRONZE_BASE_PATH}/{table_name}"
    
    # 2. CHECK FOR FILES IN MINIO (Replacing os.listdir)
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=landing_folder_key)
    
    # Filter for actual .csv files (ignoring the folder marker itself)
    files_to_process = [
        obj['Key'] for obj in response.get('Contents', []) 
        if obj['Key'].endswith('.csv')
    ]
    
    if not files_to_process:
        print(f"Skipping {table_name}: No new files in s3://{BUCKET}/{landing_folder_key}")
        continue

    print(f"Processing {table_name}: Found {len(files_to_process)} new file(s)...")

    try:
        # 3. SPARK READS DIRECTLY FROM S3
        s3a_read_path = f"s3a://{BUCKET}/{landing_folder_key}*.csv"
        
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(s3a_read_path)
            
        # Add Governance Metadata
        df_bronze = df_new \
            .withColumn("_ingested_at", current_timestamp()) \
            .withColumn("_source_file", input_file_name())

        # 4. APPEND TO BRONZE DELTA
        df_bronze.write.format("delta") \
            .mode("append") \
            .save(bronze_dest)
        
        # 5. MOVE TO ARCHIVE IN S3 (Replacing shutil.move)
        for old_key in files_to_process:
            filename = old_key.split('/')[-1]
            new_key = f"{ARCHIVE_PREFIX}/{folder_name}/{filename}"
            
            # Copy to archive prefix, then delete from landing prefix
            s3.copy_object(Bucket=BUCKET, CopySource={'Bucket': BUCKET, 'Key': old_key}, Key=new_key)
            s3.delete_object(Bucket=BUCKET, Key=old_key)
            
        print(f"SUCCESS: {table_name} appended to Bronze and archived in MinIO.")

    except Exception as e:
        print(f"ERROR: Failed to process {table_name}. Reason: {e}")

print("--- All tables processed ---")

--- Starting Cloud-Native Bronze Ingestion ---
Processing olist_orders: Found 1 new file(s)...
✅ SUCCESS: olist_orders appended to Bronze and archived in MinIO.
Processing olist_customers: Found 1 new file(s)...
✅ SUCCESS: olist_customers appended to Bronze and archived in MinIO.
Processing olist_items: Found 1 new file(s)...
✅ SUCCESS: olist_items appended to Bronze and archived in MinIO.
Processing olist_products: Found 1 new file(s)...
✅ SUCCESS: olist_products appended to Bronze and archived in MinIO.
Processing olist_payments: Found 1 new file(s)...
✅ SUCCESS: olist_payments appended to Bronze and archived in MinIO.
Processing olist_reviews: Found 1 new file(s)...
✅ SUCCESS: olist_reviews appended to Bronze and archived in MinIO.
Processing olist_sellers: Found 1 new file(s)...
✅ SUCCESS: olist_sellers appended to Bronze and archived in MinIO.
Processing olist_geolocation: Found 1 new file(s)...
✅ SUCCESS: olist_geolocation appended to Bronze and archived in MinIO.
Processing tran

In [12]:
import boto3

# 1. CONNECTION CONFIGURATION
s3 = boto3.client('s3',
    endpoint_url='http://minio:9000', # Use 'localhost:9000' if running on your host Ubuntu machine
    aws_access_key_id='admin',   # Adjust if you changed these
    aws_secret_access_key='password',
    region_name='us-east-1'
)

BUCKET = "olist-data"

# Mapping: Table Name -> Subfolder Name
pipeline_tables = {
    "olist_orders": "orders",
    "olist_customers": "customers",
    "olist_items": "items",
    "olist_products": "products",
    "olist_payments": "payments",
    "olist_reviews": "reviews",
    "olist_sellers": "sellers",
    "olist_geolocation": "geolocation",
    "translation": "translation",
    "crm_identities": "crm",
    "helpdesk_tickets": "helpdesk"
}

print("--- Creating Persistent Folders in MinIO ---")

# 2. CREATE ANCHOR FILES FOR LANDING AND ARCHIVE
zones = ["landing", "archive"]

for zone in zones:
    for _, folder_name in pipeline_tables.items():
        # Define the exact path for the hidden file
        # e.g., landing/orders/.invisible
        anchor_key = f"{zone}/{folder_name}/.invisible"
        
        try:
            # put_object creates the file directly. 
            # Body=b"" means we are uploading a completely empty (0-byte) file.
            s3.put_object(
                Bucket=BUCKET, 
                Key=anchor_key, 
                Body=b""
            )
            print(f"Created persistent folder: {zone}/{folder_name}/")
        except Exception as e:
            print(f"Failed to create {zone}/{folder_name}/. Reason: {e}")

print("--- MinIO Infrastructure Anchored Successfully ---")

--- Creating Persistent Folders in MinIO ---
✅ Created persistent folder: landing/orders/
✅ Created persistent folder: landing/customers/
✅ Created persistent folder: landing/items/
✅ Created persistent folder: landing/products/
✅ Created persistent folder: landing/payments/
✅ Created persistent folder: landing/reviews/
✅ Created persistent folder: landing/sellers/
✅ Created persistent folder: landing/geolocation/
✅ Created persistent folder: landing/translation/
✅ Created persistent folder: landing/crm/
✅ Created persistent folder: landing/helpdesk/
✅ Created persistent folder: archive/orders/
✅ Created persistent folder: archive/customers/
✅ Created persistent folder: archive/items/
✅ Created persistent folder: archive/products/
✅ Created persistent folder: archive/payments/
✅ Created persistent folder: archive/reviews/
✅ Created persistent folder: archive/sellers/
✅ Created persistent folder: archive/geolocation/
✅ Created persistent folder: archive/translation/
✅ Created persistent

In [3]:
!pip install s3fs
!pip install delta-spark==3.1.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 770.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 801.4 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 826.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 928.8 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 3.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.1/231.1 kB 5.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.3/246.3 kB 5.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2023.9.2
    Uninstalling fsspec-2023.9.2:
      Successfully uninstalle

In [1]:
import os
import uuid
import random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Setup & Configuration
LANDING_DIR = "./landing"
os.makedirs(LANDING_DIR, exist_ok=True)

NUM_CUSTOMERS = 50
NUM_SELLERS = 10
NUM_PRODUCTS = 20
NUM_ORDERS = 100
SHARED_ZIP_CODES = [random.randint(10000, 99999) for _ in range(20)]
def generate_id():
    return uuid.uuid4().hex

def random_date(start_date, end_date):
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    random_number_of_days = random.randrange(days_between_dates)
    return start_date + timedelta(days=random_number_of_days)

# 2. Generate Dimension Tables (Primary Keys)
print("Generating Dimension Tables...")

# Customers & CRM Identities
customer_ids = [generate_id() for _ in range(NUM_CUSTOMERS)]
customer_unique_ids = [generate_id() for _ in range(NUM_CUSTOMERS)]
states = ['SP', 'RJ', 'MG', 'RS', 'PR']

customers_df = pd.DataFrame({
    'customer_id': customer_ids,
    'customer_unique_id': customer_unique_ids,
    'customer_zip_code_prefix': [random.choice(SHARED_ZIP_CODES) for _ in range(NUM_CUSTOMERS)],
    'customer_city': [random.choice(['sao paulo', 'rio de janeiro', 'belo horizonte', 'curitiba']) for _ in range(NUM_CUSTOMERS)],
    'customer_state': [random.choice(states) for _ in range(NUM_CUSTOMERS)]
})

crm_df = pd.DataFrame({
    'customer_unique_id': customer_unique_ids,
    'email': [f"user_{str(uuid.uuid4())[:8]}@example.com" for _ in range(NUM_CUSTOMERS)]
})

# Sellers
seller_ids = [generate_id() for _ in range(NUM_SELLERS)]
sellers_df = pd.DataFrame({
    'seller_id': seller_ids,
    'seller_zip_code_prefix': [random.choice(SHARED_ZIP_CODES) for _ in range(NUM_SELLERS)],
    'seller_city': [random.choice(['sao paulo', 'campinas', 'florianopolis']) for _ in range(NUM_SELLERS)],
    'seller_state': [random.choice(states) for _ in range(NUM_SELLERS)]
})

# Products
product_ids = [generate_id() for _ in range(NUM_PRODUCTS)]
categories = ['cama_mesa_banho', 'esporte_lazer', 'moveis_decoracao', 'beleza_saude', 'informatica_acessorios']
products_df = pd.DataFrame({
    'product_id': product_ids,
    'product_category_name': [random.choice(categories) for _ in range(NUM_PRODUCTS)],
    'product_name_lenght': np.random.randint(20, 60, NUM_PRODUCTS),
    'product_description_lenght': np.random.randint(100, 1500, NUM_PRODUCTS),
    'product_photos_qty': np.random.randint(1, 5, NUM_PRODUCTS),
    'product_weight_g': np.random.randint(100, 5000, NUM_PRODUCTS),
    'product_length_cm': np.random.randint(15, 50, NUM_PRODUCTS),
    'product_height_cm': np.random.randint(5, 30, NUM_PRODUCTS),
    'product_width_cm': np.random.randint(10, 40, NUM_PRODUCTS)
})

# 3. Generate Fact Tables (Foreign Keys referencing Dimensions)
print("Generating Fact Tables...")

# Orders
order_ids = [generate_id() for _ in range(NUM_ORDERS)]
start = datetime(2018, 1, 1)
end = datetime(2018, 8, 31)

purchase_dates = [random_date(start, end) for _ in range(NUM_ORDERS)]
approved_dates = [d + timedelta(hours=random.randint(1, 48)) for d in purchase_dates]
carrier_dates = [d + timedelta(days=random.randint(1, 3)) for d in approved_dates]
delivered_dates = [d + timedelta(days=random.randint(2, 10)) for d in carrier_dates]

orders_df = pd.DataFrame({
    'order_id': order_ids,
    'customer_id': [random.choice(customer_ids) for _ in range(NUM_ORDERS)],
    'order_status': ['delivered'] * NUM_ORDERS, # Keeping it to 'delivered' as per your whitelist
    'order_purchase_timestamp': purchase_dates,
    'order_approved_at': approved_dates,
    'order_delivered_carrier_date': carrier_dates,
    'order_delivered_customer_date': delivered_dates,
    'order_estimated_delivery_date': [d + timedelta(days=15) for d in purchase_dates]
})

# Order Items
order_items = []
for order in order_ids:
    num_items = random.randint(1, 3)
    for i in range(num_items):
        order_items.append({
            'order_id': order,
            'order_item_id': i + 1,
            'product_id': random.choice(product_ids),
            'seller_id': random.choice(seller_ids),
            'shipping_limit_date': purchase_dates[0] + timedelta(days=5),
            'price': round(random.uniform(10.0, 500.0), 2),
            'freight_value': round(random.uniform(5.0, 50.0), 2)
        })
items_df = pd.DataFrame(order_items)

# Order Payments
payments = []
for order in order_ids:
    payments.append({
        'order_id': order,
        'payment_sequential': 1,
        'payment_type': random.choice(['credit_card', 'boleto', 'voucher', 'debit_card']),
        'payment_installments': random.randint(1, 10),
        'payment_value': round(random.uniform(20.0, 600.0), 2)
    })
payments_df = pd.DataFrame(payments)

# Order Reviews
reviews_df = pd.DataFrame({
    'review_id': [generate_id() for _ in range(NUM_ORDERS)],
    'order_id': order_ids, # 1-to-1 relationship for simplicity
    'review_score': np.random.randint(1, 6, NUM_ORDERS),
    'review_comment_title': '',
    'review_comment_message': '',
    'review_creation_date': delivered_dates,
    'review_answer_timestamp': [d + timedelta(days=random.randint(1, 3)) for d in delivered_dates]
})

# Helpdesk Tickets
helpdesk_df = pd.DataFrame({
    'ticket_id': [generate_id() for _ in range(NUM_ORDERS // 2)], # 50% of orders get a ticket
    'email': [random.choice(crm_df['email'].tolist()) for _ in range(NUM_ORDERS // 2)],
    'issue_type': [random.choice(['late_delivery', 'damaged_item', 'wrong_item', 'refund_request']) for _ in range(NUM_ORDERS // 2)],
    'satisfaction_rating': np.random.randint(1, 6, NUM_ORDERS // 2)
})

# Geolocation
geo_df = pd.DataFrame({
    'geolocation_zip_code_prefix': SHARED_ZIP_CODES * 5, # Duplicated to make 100 row,
    'geolocation_lat': [random.uniform(-33.0, 5.0) for _ in range(100)], # Approx Brazil Lat bounds
    'geolocation_lng': [random.uniform(-73.0, -34.0) for _ in range(100)], # Approx Brazil Lng bounds
    'geolocation_city': [random.choice(['sao paulo', 'rio de janeiro', 'curitiba', 'brasilia']) for _ in range(100)],
    'geolocation_state': [random.choice(states) for _ in range(100)]
})

# 4. Save Directly to MinIO Bucket
print("Uploading files directly to MinIO bucket...")

# MinIO connection details from your docker-compose.yml
storage_options = {
    "key": "admin",
    "secret": "password",
    "client_kwargs": {
        "endpoint_url": "http://minio:9000" # Docker internal network address
    }
}

# The root path in your MinIO bucket
MINIO_BASE_PATH = "s3://olist-data/landing"

# Mapping: (Folder Name, File Name, DataFrame)
files_to_save = [
    ('crm', 'crm_identities.csv', crm_df),
    ('customers', 'olist_customers_dataset.csv', customers_df),
    ('geolocation', 'olist_geolocation_dataset.csv', geo_df),
    ('helpdesk', 'helpdesk_tickets.csv', helpdesk_df),
    ('items', 'olist_order_items_dataset.csv', items_df),
    ('orders', 'olist_orders_dataset.csv', orders_df),
    ('payments', 'olist_order_payments_dataset.csv', payments_df),
    ('products', 'olist_products_dataset.csv', products_df),
    ('reviews', 'olist_order_reviews_dataset.csv', reviews_df),
    ('sellers', 'olist_sellers_dataset.csv', sellers_df)
]

for folder, filename, df in files_to_save:
    # Construct the full S3 path: s3://olist-data/landing/folder/filename.csv
    s3_path = f"{MINIO_BASE_PATH}/{folder}/{filename}"
    
    # Write directly to MinIO over the network
    df.to_csv(s3_path, index=False, storage_options=storage_options)
    print(f"Uploaded: {s3_path} ({len(df)} rows)")

print("\nDirect upload to MinIO complete! Check your MinIO UI.")

Generating Dimension Tables...
Generating Fact Tables...
Uploading files directly to MinIO bucket...
Uploaded: s3://olist-data/landing/crm/crm_identities.csv (50 rows)
Uploaded: s3://olist-data/landing/customers/olist_customers_dataset.csv (50 rows)
Uploaded: s3://olist-data/landing/geolocation/olist_geolocation_dataset.csv (100 rows)
Uploaded: s3://olist-data/landing/helpdesk/helpdesk_tickets.csv (50 rows)
Uploaded: s3://olist-data/landing/items/olist_order_items_dataset.csv (201 rows)
Uploaded: s3://olist-data/landing/orders/olist_orders_dataset.csv (100 rows)
Uploaded: s3://olist-data/landing/payments/olist_order_payments_dataset.csv (100 rows)
Uploaded: s3://olist-data/landing/products/olist_products_dataset.csv (20 rows)
Uploaded: s3://olist-data/landing/reviews/olist_order_reviews_dataset.csv (100 rows)
Uploaded: s3://olist-data/landing/sellers/olist_sellers_dataset.csv (10 rows)

Direct upload to MinIO complete! Check your MinIO UI.
